# Summary Tree Index — hierarchical summarization

A flat vector index retrieves *chunks*. A **summary tree index** goes one level up: it summarizes each leaf (document or chunk) into a compact representation, then clusters those summaries into topic groups and summarizes each group. Queries route first to the cluster level (cheap global answer) then drill into the relevant leaves (precise answer).

This is essentially LlamaIndex's `SummaryIndex` + tree-style retrieval. We implement it from scratch to show how the routing logic works.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


In [ ]:
from shared.embedders import cosine_topk, hash_embed
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa

MODEL = 'openai/gpt-4o-mini'
NS = '02-indexing/05-summary-tree-index'
DOCS = load_corpus()
QA   = [q for q in load_golden_qa() if q.source_ids]
doc_ids = [d.arxiv_id for d in DOCS]
print('docs:', len(DOCS))

## Layer 0 — leaf summaries (one call per doc)

In [ ]:
LEAF_SYS = (
    'Summarize the following research abstract in 1-2 crisp sentences. '
    'Include the method name, the key result (one metric), and the problem solved. '
    'Output ONLY the summary, no preamble.'
)
def leaf_summary(abstract):
    return complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=LEAF_SYS),
        Message(role='user', content=abstract),
    ]).content

leaf_sums = {d.arxiv_id: leaf_summary(d.abstract) for d in DOCS}
for did, s in list(leaf_sums.items())[:2]:
    print(f'{did}: {s[:80]}...')

## Layer 1 — cluster summaries

Group the leaves into 3 topic clusters by embedding the leaf summaries and running k-means.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

sum_texts = [leaf_sums[d.arxiv_id] for d in DOCS]
sum_vecs  = hash_embed(sum_texts, dims=256, seed=0)
n_clusters = 3
km = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
labels = km.fit_predict(sum_vecs)
clusters = {i: [doc_ids[j] for j, lbl in enumerate(labels) if lbl == i] for i in range(n_clusters)}
for ci, members in clusters.items():
    print(f'cluster {ci}: {members}')

In [ ]:
CLUSTER_SYS = (
    'You are given summaries of several related research papers. '
    'Write a single cluster summary (3-4 sentences) that describes the shared theme, '
    'the key methods, and the overall research direction. Neutral academic tone.'
)
def cluster_summary(cluster_doc_ids):
    block = '\n\n'.join(f'[{did}] {leaf_sums[did]}' for did in cluster_doc_ids)
    return complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=CLUSTER_SYS),
        Message(role='user', content=f'Summaries:\n{block}'),
    ]).content

cluster_sums = {ci: cluster_summary(members) for ci, members in clusters.items()}
for ci, s in cluster_sums.items():
    print(f'--- cluster {ci} ---')
    print(s[:120], '...' if len(s) > 120 else '')
    print()

## Query routing — cluster first, then leaf drill

In [ ]:
cluster_vecs = hash_embed([cluster_sums[ci] for ci in range(n_clusters)], dims=256, seed=0)

def tree_retrieve(q, k_leaf=3):
    qv = hash_embed([q], dims=256, seed=0)[0]
    # 1. Find closest cluster
    cidx, _ = cosine_topk(qv, cluster_vecs, k=1)
    best_cluster = cidx[0]
    # 2. Search within that cluster's leaf summaries
    members = clusters[best_cluster]
    member_vecs = hash_embed([leaf_sums[did] for did in members], dims=256, seed=0)
    leaf_idx, _ = cosine_topk(qv, member_vecs, k=min(k_leaf, len(members)))
    return [members[i] for i in leaf_idx]

def flat_retrieve(q, k=3):
    qv = hash_embed([q], dims=256, seed=0)[0]
    idx, _ = cosine_topk(qv, sum_vecs, k=k)
    return [doc_ids[i] for i in idx]

# Compare tree vs flat retrieval
hits_tree = hits_flat = 0
for item in QA:
    if set(tree_retrieve(item.question)) & set(item.source_ids): hits_tree += 1
    if set(flat_retrieve(item.question)) & set(item.source_ids): hits_flat += 1
print(f'Tree recall@3 : {hits_tree/len(QA):.4f}')
print(f'Flat recall@3 : {hits_flat/len(QA):.4f}')

## What you can extend

* Add a third level (corpus → cluster → document → chunk) for very large corpora.
* Use the cluster summary as context in the RAG prompt — it orients the LLM even before chunk text.
* Replace k-means with HDBSCAN for soft cluster boundaries.
* Use LlamaIndex `SummaryIndex` with `response_mode='tree_summarize'` for a production version.